# Case study 1: Policy simulation with `its2s`

ITS counterfactual on simulated daily ED encounters (`case_study1_sim_data.ipynb`), using the **Prophet + XGBoost hybrid** (`prophet_xgb`).

- **Data:** `../data/sim_ed_encounters.csv` (2022–2026)
- **Intervention:** 2025-03-01
- **Known true effect:** +12% level shift → estimates should be close to +12%
- **Analysis:** `run_single_its` with package defaults (`its2s/params.yaml`)

In [ ]:
from pathlib import Path
import logging
import warnings

import pandas as pd
from IPython.display import Image, display
from its2s import run_single_its

# Prophet/CmdStan log every bootstrap refit at INFO — keep notebook readable
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)
for _name in ("cmdstanpy", "prophet", "stan", "its2s", "xgboost", "matplotlib"):
    logging.getLogger(_name).setLevel(logging.WARNING)

DATA_PATH = Path("../data/sim_ed_encounters.csv")
OUT_DIR = Path("../outputs")
INTERVENTION = "2025-03-01"
TRUE_EFFECT_PCT = 12.0
PLOT_COLORS = ["#984136", "#c26a7a", "#ecc0a1", "#f0f0e4"]
PLOT_FONT_SIZES = {"title": 22, "axis_label": 20, "tick": 18, "legend": 18}

df = pd.read_csv(DATA_PATH, parse_dates=["ds"]).sort_values("ds").reset_index(drop=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
result = run_single_its(
    df,
    intervention_date=INTERVENTION,
    config_overrides={
        "output": {
            "plot_colors": PLOT_COLORS,
            "plot_font_sizes": PLOT_FONT_SIZES,
        },
    },
    output_dir=OUT_DIR,
)

print(result.summary())

## Results: excess post-intervention

In [ ]:
# results 

row = result.excess_table.period_excess.iloc[0]
est = row["excess_pct"]
# period CIs are on total excess (counts); convert to % for display
pct_lo = 100 * row["excess_ci_lo"] / row["total_expected"]
pct_hi = 100 * row["excess_ci_hi"] / row["total_expected"]

print(f"Simulated true effect          : +{TRUE_EFFECT_PCT:.1f}%")
print(f"Estimated attributable fraction: {est:+.2f}%  [95% CI: {pct_lo:+.2f}%, {pct_hi:+.2f}%]")
print(f"Difference (estimated − true)        : {est - TRUE_EFFECT_PCT:+.2f} pp")

In [ ]:
display(Image(filename=OUT_DIR / f"{result.model_name}_counterfactual.png"))

In [ ]:
from its2s.data_prep import prepare_splits
from its2s.outputs.plots import plot_counterfactual

splits = prepare_splits(
    df,
    INTERVENTION,
)

result.config["output"]["plot_font_sizes"] = {
    "title": 22,
    "axis_label": 20,
    "tick": 18,
    "legend": 18,
}

plot_counterfactual(
    result,
    splits,
    save_path=OUT_DIR / f"{result.model_name}_counterfactual.png",
    config=result.config,
)

display(Image(filename=OUT_DIR / f"{result.model_name}_counterfactual.png"))